Ce notebok est pensé pour être exécuté dans un environnement cloud AWS.

Vous trouverez la version locale de ce code sur https://github.com/Bright381/Projet-9.

# Démarrage de la session Spark

In [ ]:
# start pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1788888520636_0004,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# %%configure -f

# {
#   "conf": {
#       "spark.pyspark.virtualenv.enabled": "true"
#   }
# }

In [ ]:
# import sys
# print(sys.executable)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/usr/bin/python3.11

In [ ]:
# %pip --version

pip 26.0.1 from /opt/mamba/lib/python3.9/site-packages/pip (python 3.9)
Note: you may need to restart the kernel to use updated packages.


Listons les librairies installées.

In [ ]:
sc.list_packages() 

Import des librairies

In [ ]:
import pandas as pd
from PIL import Image
import numpy as np
import io
import os

from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
import tensorflow as tf
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.sql import SparkSession

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Définition des chemins

In [ ]:
PATH = 's3://oc-p9-822127610761-eu-north-1-an/'
PATH_Data = PATH+'data'
PATH_Result = PATH+'jupyter/jovyan/output_notebook/results'
CHECKPOINT_PATH="s3://oc-p9-822127610761-eu-north-1-an/jupyter/jovyan/outbook_notebook/checkpoint"
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

# Traitement des données, Transfer Learning

## Chargement des images

In [ ]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

In [ ]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

## Chargement du modèle

In [ ]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

In [ ]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

In [ ]:
new_model.summary()

Diffusion des poids du modèle à l'ensemble des workers de la session Spark

In [ ]:
brodcast_weights = sc.broadcast(new_model.get_weights())

## Fonctions de preprocessing

In [ ]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

In [ ]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

In [ ]:
# spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

In [ ]:
features_df = images.repartition(20).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

## Spark PCA pour réduction de dimension

In [ ]:
from pyspark.sql.functions import size

# Check the length of the array in the "features" column
features_df.select(
    "path", 
    size("features").alias("feature_vector_length")
).show(5, truncate=False)

In [ ]:
from pyspark.ml.functions import array_to_vector
# Convert the array<float> column into a Spark ML DenseVector
vectors_df = features_df.withColumn("features_vec", array_to_vector("features"))

# Persist features to storage to avoid repeating deep learning inference during PCA fitting
checkpoint_path = f"CHECKPOINT_PATH"  
vectors_df.write.mode("overwrite").option("compression", "gzip").format("parquet").save(checkpoint_path)

# Disable vectorized reader to prevent OOM errors, was needed for local
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")


cached_df = spark.read.parquet(checkpoint_path)

In [ ]:
from pyspark.ml.feature import PCA

pca = PCA(k=200, inputCol="features_vec", outputCol="pca_features")
pca_model = pca.fit(cached_df)

reduced_features_df = pca_model.transform(cached_df)

pip 26.0.1 from /opt/mamba/lib/python3.9/site-packages/pip (python 3.9)
Note: you may need to restart the kernel to use updated packages.


### Résultats PCA

In [ ]:
import matplotlib.pyplot as plt

explained_variance = pca_model.explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title("Cumulative Explained Variance by PCA Components")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.show()

print(f"Total variance explained: {cumulative_variance[-1]:.2%}")

# Sauvegarde des données

In [ ]:
print(PATH_Result)

In [ ]:
reduced_features_df.write.option("compression", "gzip").mode("overwrite").parquet(PATH_Result)

In [ ]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')

In [ ]:
df.head()

In [ ]:
df.loc[0,'features'].shape